## Experiment Loop

Generate all data: 30 iterations for each benchmark and parameter combination. After that, save everything to a CSV for visualization.

In [1]:
import sys
from pathlib import Path

# Define paths
REPO_ROOT = Path.cwd().parent               # Repo root (ikp-sbl)
LECTURE_EXAMPLES = REPO_ROOT / "lecture_examples"

# Add BOTH to sys.path
for path in [str(REPO_ROOT), str(LECTURE_EXAMPLES)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import pandas as pd

import lecture_examples.IPTestSuite as ts
from planners.SBL import BidirectionalSBL

In [2]:
# 1. Define environments (Task 10)
# Assuming ts.benchList is a list of benchmark objects
environments = ts.benchList

# 2. Define parameter variations (Task 13)
parameter_configs = [
    {"eta": 0.5, "epsilon": 0.05, "goal_bias": 0.5, "repair_bias": 0.6}, # Default config
    {"eta": 1.0, "epsilon": 0.05, "goal_bias": 0.5, "repair_bias": 0.6}, # Only changed eta
    {"eta": 0.5, "epsilon": 0.05, "goal_bias": 0.7, "repair_bias": 0.6}, # Changed only goal_bias
    {"eta": 1.0, "epsilon": 0.2,  "goal_bias": 0.3, "repair_bias": 0.6}, # Changes in eta, epsilon and goal_bias
    {"eta": 1.2, "epsilon": 0.2,  "goal_bias": 0.8, "repair_bias": 0.3}, # Changes in all
    {"eta": 0.5, "epsilon": 0.05, "goal_bias": 0.5, "repair_bias": 0.05}, # Changed only repair_bias (very low)
    {"eta": 0.5, "epsilon": 0.5, "goal_bias": 0.5, "repair_bias": 0.6},  # Changed only epsilon (very high)
    {"eta": 0.5, "epsilon": 0.05, "goal_bias": 0.1, "repair_bias": 0.8},  # Changes in goal_bias and repair_bias
    {"eta": 0.5, "epsilon": 0.05, "goal_bias": 0.0, "repair_bias": 0.6}, # Pure exploration (0% goal bias)
    {"eta": 0.5, "epsilon": 0.05, "goal_bias": 0.95, "repair_bias": 0.6},# Aggressive greed (95% goal bias)
    {"eta": 0.5, "epsilon": 0.01, "goal_bias": 0.5, "repair_bias": 0.6}  # Very fine adaptive resolution (only epsilon)
    ]

experiment_data = []
total_runs = 30

# 3. The Nested Experiment Loop
for bench in environments:
    scene_name = bench.name  # Extract the name directly from the benchmark object
    print(f"Starting experiments for scene: {scene_name}...")
    
    for config_idx, config in enumerate(parameter_configs):
        for run_id in range(total_runs):
            
            # Setup the planner with the specific config
            planner = BidirectionalSBL(bench.collisionChecker, config=config) 
            
            # Run the planner
            path = planner.planPath(bench.startList, bench.goalList, config)
            
            # 4. Extract all 11 metrics from your PlannerStats
            stats = planner.stats
            
            # Build the row dictionary
            run_result = {
                # Setup info
                "Scene": scene_name,
                "Run_ID": run_id,
                "Config_Idx": config_idx,
                "Eta": config["eta"],
                "Epsilon": config["epsilon"],
                "Goal_Bias": config["goal_bias"],       
                "Repair_Bias": config["repair_bias"],   
                
                # Output Metrics (Task 13)
                "Success": stats.success,
                "Plan_Time": stats.planning_time,
                "Time_First_Candidate": stats.time_to_first_candidate,
                "Time_First_Valid": stats.time_to_first_valid_path,
                "Nodes_Start_Tree": stats.total_nodes_start_tree,
                "Nodes_Goal_Tree": stats.total_nodes_goal_tree,
                "Edges_Unknown": stats.edges_unknown,
                "Edges_Valid": stats.edges_valid,
                "Edges_Invalid": stats.edges_invalid,
                "Point_Checks": stats.point_collision_tests,
                "Line_Checks": stats.line_tests,
                "Aborted_Adaptive_Tests": stats.aborted_adaptive_tests,
                "Final_Path_Length": stats.path_length,
                "Candidate_Paths_Count": stats.candidate_paths_checked
            }
            
            experiment_data.append(run_result)

# 5. Convert to Pandas DataFrame for easy analysis
df_results = pd.DataFrame(experiment_data)

# SAVE THE DATA IMMEDIATELY
df_results.to_csv("sbl_benchmark_results.csv", index=False)
print("Experiment finished and data saved to CSV!")

Starting experiments for scene: Trap...


  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 856.82it/s]


Starting experiments for scene: Fat bottleneck...


  0%|          | 0/10 [00:00<?, ?it/s]


Starting experiments for scene: Open Space...


  0%|          | 0/10 [00:00<?, ?it/s]


Starting experiments for scene: Random Map...


 20%|██        | 2/10 [00:00<00:00, 100.72it/s]

Experiment finished and data saved to CSV!


In [3]:
df_results.head()

,Scene,Run_ID,Config_Idx,Eta,Epsilon,Goal_Bias,Repair_Bias,Success,Plan_Time,Time_First_Candidate,...,Nodes_Start_Tree,Nodes_Goal_Tree,Edges_Unknown,Edges_Valid,Edges_Invalid,Point_Checks,Line_Checks,Aborted_Adaptive_Tests,Final_Path_Length,Candidate_Paths_Count
0,Trap,0,0,0.5,0.05,0.5,0.6,False,0.019687,0.016825,...,22,16,13,15,8,73,25,10,0.0,10
1,Trap,1,0,0.5,0.05,0.5,0.6,False,0.018813,0.001217,...,40,62,59,33,8,318,43,10,0.0,10
2,Trap,2,0,0.5,0.05,0.5,0.6,False,0.013443,0.010588,...,34,35,42,20,5,208,30,10,0.0,10
3,Trap,3,0,0.5,0.05,0.5,0.6,False,0.022095,0.001859,...,61,50,60,42,7,461,52,10,0.0,10
4,Trap,4,0,0.5,0.05,0.5,0.6,False,0.020823,0.001046,...,73,44,83,24,8,353,34,10,0.0,10
